# Capacità ed early stopping



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Keras registra loss di training e validazione durante le epoche. Impareremo a interpretare queste **learning curve**, riconoscere underfitting e overfitting e correggerli.

# Interpretare le curve
Possiamo distinguere nei dati **segnale** e **rumore**. Il segnale comprende relazioni che si ripetono e aiutano a generalizzare; il rumore comprende fluttuazioni e associazioni occasionali specifiche del campione.

L'addestramento minimizza la loss sul training set. Per valutare il modello servono dati nuovi: ripassa la [validazione](https://www.kaggle.com/dansbecker/model-validation) se necessario. Disegniamo quindi entrambe le curve, epoca per epoca.

![Loss di training e validazione](https://storage.googleapis.com/kaggle-media/learn/images/tHiVFnM.png)

La training loss può diminuire sia apprendendo segnale sia memorizzando rumore. La validation loss tende invece a migliorare quando il modello apprende relazioni trasferibili. Un divario crescente fra le curve può segnalare overfitting.

Vorremmo apprendere tutto il segnale senza rumore, ma nella pratica cerchiamo un compromesso. Finché i benefici prevalgono, la validation loss scende; quando il rumore domina può risalire.

![Underfitting e overfitting](https://storage.googleapis.com/kaggle-media/learn/images/eUF6mfo.png)

**Underfitting** significa non aver appreso abbastanza relazioni utili; **overfitting** significa essersi adattati troppo a particolarità del training. Conta la capacità di generalizzare, non soltanto una bassa training loss.

# Capacità del modello
La **capacity** indica quanto complesse siano le relazioni rappresentabili. Dipende dal numero di neuroni e dalle connessioni. Se il modello è troppo semplice, prova ad aumentarla:

* Più **largo**: più unità per strato.
* Più **profondo**: più strati.

La scelta dipende dai dati. Il tutorial associa maggiore larghezza a relazioni più lineari e maggiore profondità a composizioni non lineari; non è una regola universale.

```python
model = keras.Sequential([
    layers.Dense(16, activation='relu'),
    layers.Dense(1),
])
wider = keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
])
deeper = keras.Sequential([
    layers.Dense(16, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1),
])
```

# Early stopping
Possiamo interrompere il training quando la validation loss smette di migliorare. È l'**early stopping**.

![Arresto vicino al minimo della validation loss](https://storage.googleapis.com/kaggle-media/learn/images/eP0gppr.png)

Ripristinare i pesi del miglior risultato evita di conservare quelli delle ultime epoche, potenzialmente sovradattati. Possiamo impostare molte epoche e lasciare che il controllo fermi il training; riduciamo così anche il rischio di interromperlo troppo presto.

## Callback in Keras
Una **callback** viene eseguita in momenti prestabiliti del training. `EarlyStopping` controlla le misure alla fine di ogni epoca. Keras offre [callback già definite](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks) e permette [callback personalizzate](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/LambdaCallback).


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    min_delta=0.001, # Variazione minima da considerare come miglioramento.
    patience=20, # Numero di epoche da attendere prima di interrompere il training.
    restore_best_weights=True,
)

I parametri significano: «Se per 20 epoche la validation loss non migliora di almeno 0,001, ferma il training e ripristina i pesi migliori». La tolleranza aiuta a non confondere normali oscillazioni dei batch con un peggioramento stabile.

La callback si passa a **`fit`**. Continuiamo l'esempio sul vino: aumentiamo la capacità e aggiungiamo l'early stopping. Prima ripetiamo la preparazione dei dati.


In [ ]:

import pandas as pd
from IPython.display import display

red_wine = pd.read_csv('../input/dl-course-data/red-wine.csv')

# Suddividi i dati in training e validazione.
df_train = red_wine.sample(frac=0.7, random_state=0)
df_valid = red_wine.drop(df_train.index)
display(df_train.head(4))

# Riporta i valori nell'intervallo [0, 1].
max_ = df_train.max(axis=0)
min_ = df_train.min(axis=0)
df_train = (df_train - min_) / (max_ - min_)
df_valid = (df_valid - min_) / (max_ - min_)

# Separa feature e target.
X_train = df_train.drop('quality', axis=1)
X_valid = df_valid.drop('quality', axis=1)
y_train = df_train['quality']
y_valid = df_valid['quality']

Aumentiamo la capacità con una rete piuttosto grande; la callback fermerà il training quando la validazione non migliorerà più.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, callbacks

early_stopping = callbacks.EarlyStopping(
    min_delta=0.001, # Variazione minima da considerare come miglioramento.
    patience=20, # Numero di epoche da attendere prima di interrompere il training.
    restore_best_weights=True,
)

model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=[11]),
    layers.Dense(512, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(1),
])
model.compile(
    optimizer='adam',
    loss='mae',
)

Passa la callback a **`fit`** dentro una lista: puoi usarne più di una. Imposta un massimo di epoche sufficientemente alto.


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=256,
    epochs=500,
    callbacks=[early_stopping], # Inserisci le callback in una lista.
    verbose=0,  # Disattiva il log del training.
)

history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot();
print("Minimum validation loss: {}".format(history_df['val_loss'].min()))

Nell'esempio Keras termina prima delle 500 epoche previste.

Ora **[prevedi la popolarità di un brano](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex4.html)** con il dataset *Spotify*.
